# K10 · Experiment queue on 2 GPUs (steps 3: OOD, ablations, ours on the baseline images)

Runs the job files below with the official code / our core, **both T4s in parallel** (each job is split into
image chunks; a free GPU takes the next chunk). Same outputs and the same config hashes as the Colab runner.

| Job file | What | Samples | ≈ T4-hours |
|---|---|---|---|
| `phase7_ood_gpu.yaml` | Set B (24 synthetic, 128 px) × 20, Set A (24 real OOD, 256 px) × 30 | 1 230 | ≈ 7 |
| `phase9_ablations.yaml` | AB1–AB4, AB7 on the 6-image subset × 10 | 900 | ≈ 8 |
| `phase8_ours_on_baseline_images.yaml` | our method on the SinDDM / LoRA images | 240 | ≈ 1.5 |

≈ 16 T4-hours = **≈ 8 hours on 2 GPUs**: one session. Progress: one bar per job file (samples finished).

**How to run (Kaggle):**
1. Right panel → *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
2. Top right **Save Version** → *Save & Run All (Commit)* → Save. It runs in the background (you can close the tab);
   watch it under *View Active Events* / the version's **Logs** (progress lines every 2 min).
3. When it finishes, the result is the version's **Output**: `cs787_k10.zip`.
4. **Not finished?** (it stops itself before Kaggle's 12-h limit) → *Add Input* → *Your Work* → this notebook's
   latest version → then *Save Version* again. It continues where it stopped.

In [ ]:
import time
T_START = time.time()                      # session clock: work stops before SESSION_HOURS
NOTEBOOK = "K10_run_jobs"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
SESSION_HOURS = 11.0                       # Kaggle stops sessions at 12 h; we stop starting new work at this point

JOB_FILES = ["configs/jobs/phase7_ood_gpu.yaml",
             "configs/jobs/phase9_ablations.yaml",
             "configs/jobs/phase8_ours_on_baseline_images.yaml"]

In [ ]:
import os, sys, subprocess, shutil, glob, json
if not os.path.isdir("/kaggle/working"):
    raise SystemExit("This notebook is written for Kaggle (kaggle.com/code).")
WORK, TMP = "/kaggle/working", "/tmp/cs787"
PROJECT = f"{TMP}/project"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(TMP, exist_ok=True)
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, PROJECT],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit("Could not download the code from GitHub. Turn Internet ON: right panel -> Session options -> "
                     "Internet (needs a phone-verified Kaggle account), then run again.\n" + r.stderr)
sys.path.insert(0, f"{PROJECT}/src")
os.environ["PYTHONPATH"] = f"{PROJECT}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")
from sidx.kaggle import gather_inputs, unpack_own, pack
from sidx.progress import Bars
print("code:", subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
gather_inputs()          # results of earlier versions attached under /kaggle/input (resume)

In [ ]:
import torch
N_GPUS = torch.cuda.device_count()
if N_GPUS == 0:
    raise SystemExit("No GPU. Right panel -> Session options -> Accelerator -> 'GPU T4 x2', then run again.")
cap = torch.cuda.get_device_capability(0)
if f"sm_{cap[0]}{cap[1]}" not in torch.cuda.get_arch_list():
    raise SystemExit(f"{torch.cuda.get_device_name(0)} is not supported by this PyTorch build: choose 'GPU T4 x2' (not P100).")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", N_GPUS, "GPU(s) -> all of them are used")

In [ ]:
SID_ROOT = f"{TMP}/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)
subprocess.run(["git", "-C", SID_ROOT, "checkout", "-q", SID_COMMIT], check=True)
!pip install -q -r {PROJECT}/env/requirements-colab.txt
!pip install -q --no-deps -e {SID_ROOT}
!pip install -q --no-deps -e {PROJECT}
os.environ["SID_ROOT"] = SID_ROOT
sys.path.insert(0, SID_ROOT)
import efficient_sid, sidx
print("efficient_sid OK | sidx", sidx.__version__)

In [ ]:
# natural images (balloons, forest, ...) are third-party and not in the repo: fetch + check sha256 against the manifests
r = subprocess.run([sys.executable, "scripts/prepare_natural.py", "--verify", "--sets", "uncond/sinddm15", "--sid-root", SID_ROOT],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
assert r.returncode == 0, "data check failed (see above)"

## Plan: what is still pending (resumes automatically)

In [ ]:
from sidx.gpupool import job_tasks, run_pool
unpack_own()
OUT_ROOT = f"{WORK}/outputs"
DEADLINE = T_START + SESSION_HOURS * 3600 - 20 * 60        # keep 20 min to zip the results
tasks, poll = job_tasks(JOB_FILES, OUT_ROOT, PROJECT, {"SID_ROOT": SID_ROOT}, N_GPUS, deadline=DEADLINE)
for exp, (d, t) in poll().items():
    print(f"{exp:22s} {d:5d}/{t} samples already done")
print(len(tasks), "tasks to run:", ", ".join(t.label for t in tasks[:12]), "..." if len(tasks) > 12 else "")

## Run on all GPUs (live bars; per-task logs in /kaggle/working/logs)

In [ ]:
try:
    run_pool(tasks, poll, n_gpus=N_GPUS, log_dir=f"{WORK}/logs/{NOTEBOOK}", deadline=DEADLINE, unit="sample")
finally:
    for exp, (d, t) in poll().items():
        print(f"{exp:22s} {d:5d}/{t} samples" + ("  DONE" if d == t else "  -> not finished: attach this output and run again"))
    pack("k10", kinds=("outputs",))

In [ ]:
os.makedirs(f"{WORK}/logs", exist_ok=True)
!pip freeze > {WORK}/logs/{NOTEBOOK}_pip_freeze.txt
print("finished; output files:", sorted(os.listdir(WORK)))